# MalthusJAX Level 2: Injection-Mode Crossover Operators Tutorial

This notebook demonstrates the **injection-mode crossover operators** in MalthusJAX, which implement genetic recombination with pre-allocated noise tensors and explicit RNG management.

## What is Injection-Mode Crossover?

**Injection Mode** differs from fused-mode operators in how they handle randomness:
- **Fused Mode**: Generates noise on-the-fly during recombination (tight RNG-arithmetic coupling)
- **Injection Mode**: Pre-generates ALL noise tensors upfront with shape `(input_length, num_offspring, ...)`

### Key Advantages
**Deterministic & Reproducible**: Full noise tensor is materialized once  
**Correlated Noise Patterns**: Can generate noise with cross-sample correlations  
**Easier Debugging**: Noise is captured and can be inspected/replayed  
**Clear Separation**: RNG logic fully separated from arithmetic logic

### Trade-offs
**Memory Usage**: Materializes full `(input_length × num_offspring × gene_dim)` noise tensor  
**Reshape/Transpose Overhead**: Shape manipulations may trigger copies in XLA  
**Vectorization Friendly**: One vmap call per operator = optimal fusion

## MalthusJAX 3-Tier Paradigm (Injection Variant)

1. **Tier 3 (Single RNG)**: Receives ONE PRNG key and orchestrates bulk operations
2. **Tier 2 (Noise Generation)**: Splits key internally, generates full `(input_length, num_offspring, ...)` tensors
3. **Tier 1 (Pure Arithmetic)**: Deterministic recombination using pre-generated noise

```python
# Same usage pattern as fused-mode, but with injection operators
from malthusjax.operators.crossover.uniform_crossover import UniformCrossover_injection
op = UniformCrossover_injection(num_offspring=2)
op = op.set_input_length(pop_size)  # Static contract
noise = op._generate_noise(key, config)  # Shape: (pop_size, 2, ...)
offspring = op(key, p1_pop, p2_pop, config)  # Vectorized crossover
```

In [1]:
import time
from typing import Any, Tuple, cast

import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np

from malthusjax.core.genome.real_genome import RealGenome, RealGenomeConfig, RealPopulation
from malthusjax.operators.crossover.real import (
    BinomialCrossover_injection,
    BlendCrossover,
    BlendCrossover_injection,
    SimulatedBinaryCrossover_injection,
    UniformCrossover_injection,
)

print(f"JAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend()}")
print(f"Available devices: {jax.devices()}")

# Set random seed for reproducibility
key = jr.PRNGKey(42)

JAX version: 0.8.0
JAX backend: cpu
Available devices: [CpuDevice(id=0)]


## Part 1: Uniform Crossover (Injection Mode)

**UniformCrossover_injection** pre-generates Bernoulli masks with shape `(input_length, num_offspring, genome_dim)` and applies them across all pairs and offspring simultaneously.

In [2]:
real_config = RealGenomeConfig(
    shape=(5,),
    bounds=(-10.0, 10.0),
    dtype=jnp.float32
)

pop_size = 10
key, subkey1, subkey2 = jr.split(key, 3)

p1_pop = RealPopulation.init_random(subkey1, real_config, pop_size)
p2_pop = RealPopulation.init_random(subkey2, real_config, pop_size)

print(f"Parent Population 1: {len(p1_pop)} individuals")
print(f"Parent Population 2: {len(p2_pop)} individuals")
print(f"Genome dimension: {real_config.shape[0]}")

Parent Population 1: 10 individuals
Parent Population 2: 10 individuals
Genome dimension: 5


In [3]:
uniform_inj = UniformCrossover_injection(num_offspring=2, crossover_rate=0.5)
uniform_inj = uniform_inj.set_input_length(pop_size)

print("\n" + "="*60)
print("Uniform Crossover (Injection Mode)")
print("="*60)

key, subkey = jr.split(key)
noise_masks = uniform_inj._generate_noise(subkey, real_config)

print(f"\nNoise mask shape: {noise_masks.shape}")
print(f"Expected: (pop_size={pop_size}, num_offspring=2, genes={real_config.shape[0]})")
print("Noise generation is DETERMINISTIC - same key produces same masks")

key2, subkey2 = jr.split(key)
noise_masks_2 = uniform_inj._generate_noise(subkey2, real_config)
print(f"Masks are identical: {jnp.allclose(noise_masks, noise_masks_2)}")


Uniform Crossover (Injection Mode)

Noise mask shape: (20, 5)
Expected: (pop_size=10, num_offspring=2, genes=5)
Noise generation is DETERMINISTIC - same key produces same masks
Masks are identical: False


In [4]:
print("\nInheritance Analysis:")
print("-" * 50)

for rate in [0.3, 0.5, 0.7]:
    op = UniformCrossover_injection(num_offspring=1, crossover_rate=rate)
    op = op.set_input_length(pop_size)

    key, subkey = jr.split(key)
    noise = op._generate_noise(subkey, real_config)

    # noise may be 2D (input_length, genes) when num_offspring==1,
    # or 3D (input_length, num_offspring, genes). Normalize to 3D shape.
    if noise.ndim == 2:
        mask = noise[:, jnp.newaxis, :]
    else:
        mask = noise

    # Count genes from P1 vs P2 across first offspring
    from_p1_ratio = jnp.mean(1.0 - mask[:, 0, :])  # (1 - mask) = from P1

    print(f"Rate={rate}: Expected ~{1-rate:.1%} from P1, Got {float(from_p1_ratio):.1%}")


Inheritance Analysis:
--------------------------------------------------
Rate=0.3: Expected ~70.0% from P1, Got 64.0%
Rate=0.5: Expected ~50.0% from P1, Got 50.0%
Rate=0.7: Expected ~30.0% from P1, Got 28.0%


## Part 2: Blend Crossover (Injection Mode)

**BlendCrossover_injection** pre-generates `(should_cross, random_samples)` tuples for each (pair, offspring) combination and vectorizes the blend arithmetic.

In [5]:
# Create diverse parent populations for blend analysis
key, subkey1, subkey2 = jr.split(key, 3)

# Parents at different distances
p1_pop_blend = RealPopulation.init_random(
    subkey1,
    real_config.replace(bounds=(-10.0, -2.0)),
    pop_size
)
p2_pop_blend = RealPopulation.init_random(
    subkey2,
    real_config.replace(bounds=(2.0, 10.0)),
    pop_size
)

print("Blend Crossover (Injection Mode)")
print("="*60)

alpha_values = [0.0, 0.5, 1.0]

for alpha in alpha_values:
    blend_inj = BlendCrossover_injection(num_offspring=1, alpha=alpha, crossover_rate=0.9)
    blend_inj = blend_inj.set_input_length(pop_size)

    key, subkey = jr.split(key)
    should_cross, random_vals = blend_inj._generate_noise(subkey, real_config)

    print(f"\nAlpha = {alpha}:")
    print(f"  should_cross shape: {should_cross.shape}")
    print(f"  random_samples shape: {random_vals.shape}")
    print(f"  Crossover rate achieved: {jnp.mean(should_cross):.1%}")

Blend Crossover (Injection Mode)

Alpha = 0.0:
  should_cross shape: (10,)
  random_samples shape: (10, 5)
  Crossover rate achieved: 70.0%

Alpha = 0.5:
  should_cross shape: (10,)
  random_samples shape: (10, 5)
  Crossover rate achieved: 80.0%

Alpha = 1.0:
  should_cross shape: (10,)
  random_samples shape: (10, 5)
  Crossover rate achieved: 80.0%


In [ ]:
# Visualize blend exploration with injection mode
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, alpha in enumerate(alpha_values):
    blend_inj = BlendCrossover_injection(num_offspring=20, alpha=alpha, crossover_rate=1.0)
    blend_inj = blend_inj.set_input_length(1)  # Single pair, many offspring

    key, subkey = jr.split(key)

    # Take first parent pair
    p1_single = RealPopulation(
        genes=RealGenome(values=p1_pop_blend.genes.values[0:1]),
        fitness=jnp.array([-jnp.inf]),
        config=real_config
    )
    p2_single = RealPopulation(
        genes=RealGenome(values=p2_pop_blend.genes.values[0:1]),
        fitness=jnp.array([-jnp.inf]),
        config=real_config
    )

    # Perform injection crossover
    offspring_pop = blend_inj(subkey, p1_single, p2_single, real_config)

    # Analyze first gene distribution
    first_gene = offspring_pop.genes.values[:, 0]

    # Parent values
    p1_val = p1_pop_blend.genes.values[0, 0]
    p2_val = p2_pop_blend.genes.values[0, 0]

    axes[idx].hist(first_gene, bins=15, alpha=0.7, color='blue', density=True)
    axes[idx].axvline(p1_val, color='red', linestyle='--', linewidth=2, label='Parent 1')
    axes[idx].axvline(p2_val, color='green', linestyle='--', linewidth=2, label='Parent 2')
    axes[idx].set_title(f'Blend α={alpha} (20 offspring)')
    axes[idx].set_xlabel('First Gene Value')
    axes[idx].set_ylabel('Density')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\nObservations:")
print("- α=0.0: Offspring strictly within parental bounds (conservative)")
print("- α=0.5: Offspring can exceed bounds (exploratory)")
print("- α=1.0: Maximum exploration with extended intervals")

## Part 3: Simulated Binary Crossover (Injection Mode)

**SimulatedBinaryCrossover_injection** pre-generates `(should_cross, u, swap_mask)` noise blocks, enabling distinct children per offspring call.

In [ ]:
# Demonstrate SBX with injection mode
eta_values = [2.0, 10.0, 30.0]

print("\nSimulated Binary Crossover (Injection Mode)")
print("="*60)

for eta in eta_values:
    sbx_inj = SimulatedBinaryCrossover_injection(num_offspring=2, eta=eta, crossover_rate=0.9)
    sbx_inj = sbx_inj.set_input_length(pop_size)

    key, subkey = jr.split(key)
    should_cross, u_vals, swap_masks = sbx_inj._generate_noise(subkey, real_config)

    # Normalize noise shapes to (input_length, num_offspring, genes) where appropriate.
    genes = int(real_config.shape[0])
    expected_n = int(sbx_inj.input_length * sbx_inj.num_offspring)

    # should_cross: (n,) -> (input_length, num_offspring)
    if should_cross.ndim == 1 and should_cross.shape[0] == expected_n:
        should_cross = should_cross.reshape((sbx_inj.input_length, sbx_inj.num_offspring))
    elif should_cross.ndim == 0:
        should_cross = jnp.full((sbx_inj.input_length, sbx_inj.num_offspring), bool(should_cross))

    # u_vals: (n, genes) -> (input_length, num_offspring, genes)
    if u_vals.ndim == 2 and u_vals.shape[0] == expected_n:
        u_vals = u_vals.reshape((sbx_inj.input_length, sbx_inj.num_offspring, genes))
    elif u_vals.ndim == 1 and u_vals.shape[0] == expected_n * genes:
        u_vals = u_vals.reshape((sbx_inj.input_length, sbx_inj.num_offspring, genes))

    # swap_masks: (n, genes) or (input_length, genes) -> (input_length, num_offspring, genes)
    if swap_masks.ndim == 2 and swap_masks.shape[0] == expected_n:
        swap_masks = swap_masks.reshape((sbx_inj.input_length, sbx_inj.num_offspring, genes))
    elif swap_masks.ndim == 2 and swap_masks.shape[0] == sbx_inj.input_length:
        swap_masks = swap_masks[:, jnp.newaxis, :]

    print(f"\nEta = {eta}:")
    print(f"  should_cross: {should_cross.shape}")
    print(f"  u (spread factor): {u_vals.shape}")
    print(f"  swap_mask (child selection): {swap_masks.shape}")

    # Verify per-offspring distinctness (works for num_offspring >= 2)
    if sbx_inj.num_offspring >= 2:
        offspring_1_swaps = swap_masks[:, 0, :]
        offspring_2_swaps = swap_masks[:, 1, :]
        same_swaps = jnp.mean(offspring_1_swaps == offspring_2_swaps)
        print(f"  Distinctness: {(1 - float(same_swaps)):.1%} of genes swapped differently per offspring")
    else:
        print("  Only one offspring per pair; distinctness check skipped.")

In [ ]:
# Visualize SBX offspring distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, eta in enumerate(eta_values):
    sbx_inj = SimulatedBinaryCrossover_injection(num_offspring=50, eta=eta, crossover_rate=1.0)
    sbx_inj = sbx_inj.set_input_length(1)  # Single pair

    key, subkey = jr.split(key)

    p1_single = RealPopulation(
        genes=RealGenome(values=p1_pop_blend.genes.values[0:1]),
        fitness=jnp.array([-jnp.inf]),
        config=real_config
    )
    p2_single = RealPopulation(
        genes=RealGenome(values=p2_pop_blend.genes.values[0:1]),
        fitness=jnp.array([-jnp.inf]),
        config=real_config
    )

    offspring_pop = sbx_inj(subkey, p1_single, p2_single, real_config)
    first_gene = offspring_pop.genes.values[:, 0]

    p1_val = p1_pop_blend.genes.values[0, 0]
    p2_val = p2_pop_blend.genes.values[0, 0]
    parent_mean = 0.5 * (p1_val + p2_val)

    axes[idx].hist(first_gene, bins=20, alpha=0.7, color='purple', density=True)
    axes[idx].axvline(p1_val, color='red', linestyle='--', linewidth=2, label='Parent 1')
    axes[idx].axvline(p2_val, color='green', linestyle='--', linewidth=2, label='Parent 2')
    axes[idx].axvline(parent_mean, color='orange', linestyle='-', linewidth=2, label='Midpoint')
    axes[idx].set_title(f'SBX η={eta} (50 offspring)')
    axes[idx].set_xlabel('First Gene Value')
    axes[idx].set_ylabel('Density')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Low η (2.0): High spread (exploratory)")
print("- Medium η (10.0): Moderate spread (balanced)")
print("- High η (30.0): Low spread (conservative, near parents)")

## Part 4: Binomial Crossover (Injection Mode)

**BinomialCrossover_injection** pre-generates selection masks for Differential Evolution trial vector construction.

In [ ]:
binomial_inj = BinomialCrossover_injection(num_offspring=1, crossover_rate=0.8)
binomial_inj = binomial_inj.set_input_length(pop_size)

print("\nBinomial Crossover (Injection Mode) - for DE")
print("="*60)

key, subkey = jr.split(key)
cross_masks = binomial_inj._generate_noise(subkey, real_config)

print(f"\nCross-mask shape: {cross_masks.shape}")
print(f"Expected: (pop_size={pop_size}, num_offspring=1, genes={real_config.shape[0]})")

mask = cross_masks[:, jnp.newaxis, :]


from_mutant = jnp.mean(mask[:, 0, :])  # Proportion taking from mutant (P1)
from_target = 1.0 - from_mutant  # Proportion from target (P2)

print(f"\nGenes from Mutant (P1): {float(from_mutant):.1%}")
print(f"Genes from Target (P2): {float(from_target):.1%}")
print(f"Expected: ~{0.8:.1%} from mutant")


Binomial Crossover (Injection Mode) - for DE

Cross-mask shape: (10, 5)
Expected: (pop_size=10, num_offspring=1, genes=5)

Genes from Mutant (P1): 80.0%
Genes from Target (P2): 20.0%
Expected: ~80.0% from mutant


## Part 5: Fused vs Injection Performance Comparison

Let's benchmark injection-mode vs fused-mode operators on different population sizes.

In [9]:
population_sizes = [10, 50, 100, 500]
results_fused = {}
results_injection = {}

print("\nPerformance Comparison: Fused vs Injection")
print("="*70)

for pop_size in population_sizes:
    key, k1, k2 = jr.split(key, 3)
    p1_pop = RealPopulation.init_random(k1, real_config, pop_size)
    p2_pop = RealPopulation.init_random(k2, real_config, pop_size)

    # FUSED MODE benchmark
    fused_op = BlendCrossover(num_offspring=1, alpha=0.5, crossover_rate=0.9)
    fused_op = fused_op.set_input_length(pop_size)

    key, subkey = jr.split(key)
    start = time.time()
    for _ in range(10):
        num_keys = fused_op.num_keys(p1_pop.genes.values.shape)
        keys = jr.split(subkey, num_keys)
        _ = fused_op(keys, p1_pop, p2_pop, real_config)
    fused_time = time.time() - start

    # INJECTION MODE benchmark
    injection_op = BlendCrossover_injection(num_offspring=1, alpha=0.5, crossover_rate=0.9)
    injection_op = injection_op.set_input_length(pop_size)

    key, subkey = jr.split(key)
    start = time.time()
    for _ in range(10):
        keys = jr.split(subkey, 1)  # Only 1 key needed
        _ = injection_op(keys, p1_pop, p2_pop, real_config)
    injection_time = time.time() - start

    results_fused[pop_size] = fused_time
    results_injection[pop_size] = injection_time

    ratio = fused_time / injection_time
    print(f"Pop={pop_size:3d} | Fused: {fused_time:.4f}s | Injection: {injection_time:.4f}s | Ratio: {ratio:.2f}x")


Performance Comparison: Fused vs Injection
Pop= 10 | Fused: 0.3956s | Injection: 0.0871s | Ratio: 4.54x
Pop= 50 | Fused: 0.4445s | Injection: 0.1217s | Ratio: 3.65x
Pop=100 | Fused: 0.3855s | Injection: 0.1205s | Ratio: 3.20x
Pop=500 | Fused: 0.4510s | Injection: 0.1123s | Ratio: 4.02x


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

pops = list(results_fused.keys())
fused_times = list(results_fused.values())
injection_times = list(results_injection.values())

x = np.arange(len(pops))
width = 0.35

ax.bar(x - width/2, fused_times, width, label='Fused Mode', alpha=0.8)
ax.bar(x + width/2, injection_times, width, label='Injection Mode', alpha=0.8)

ax.set_xlabel('Population Size')
ax.set_ylabel('Time (seconds) - 10 iterations')
ax.set_title('Fused vs Injection Mode: BlendCrossover Performance')
ax.set_xticks(x)
ax.set_xticklabels(pops)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nPerformance Notes:")
print("- Injection mode: One key splitting, then parallel noise generation")
print("- Fused mode: Per-pair key splitting, inline RNG + arithmetic")
print("- XLA loves both patterns but for different reasons!")

## Part 6: Memory Analysis - Injection Mode

Analyze the memory footprint of noise tensor materialization in injection mode.

In [10]:
print("\nMemory Analysis: Noise Tensor Materialization")
print("="*60)

for pop_size in [10, 100, 1000]:
    num_offspring = 2
    gene_dim = 100

    total_elements = pop_size * num_offspring * gene_dim

    # Assuming float32 (4 bytes) per element
    bytes_float32 = total_elements * 4
    mb = bytes_float32 / (1024 * 1024)

    print(f"\nPop={pop_size}, Offspring={num_offspring}, Genes={gene_dim}")
    print(f"  Total elements: {total_elements:,}")
    print(f"  Memory (float32): {mb:.2f} MB")

    # For BlendCrossover: need (should_cross, random_samples) = 2 tensors
    blend_mb = mb * 2
    print(f"  Memory (Blend = 2 tensors): {blend_mb:.2f} MB")

    # For SBX: need (should_cross, u, swap_mask) = 3 tensors
    sbx_mb = mb * 3
    print(f"  Memory (SBX = 3 tensors): {sbx_mb:.2f} MB")


Memory Analysis: Noise Tensor Materialization

Pop=10, Offspring=2, Genes=100
  Total elements: 2,000
  Memory (float32): 0.01 MB
  Memory (Blend = 2 tensors): 0.02 MB
  Memory (SBX = 3 tensors): 0.02 MB

Pop=100, Offspring=2, Genes=100
  Total elements: 20,000
  Memory (float32): 0.08 MB
  Memory (Blend = 2 tensors): 0.15 MB
  Memory (SBX = 3 tensors): 0.23 MB

Pop=1000, Offspring=2, Genes=100
  Total elements: 200,000
  Memory (float32): 0.76 MB
  Memory (Blend = 2 tensors): 1.53 MB
  Memory (SBX = 3 tensors): 2.29 MB


## Part 7: Custom Injection Crossover

Implement a custom injection-mode crossover demonstrating the full pattern.

In [14]:
import chex
from flax import struct

from malthusjax.operators.base_injection import BaseCrossover_injection


@struct.dataclass
class AdaptiveAlphaCrossover_injection(
    BaseCrossover_injection[RealGenome, RealGenomeConfig, RealPopulation]
):
    """
    Custom Adaptive Alpha Blend Crossover (Injection Mode).
    
    Demonstrates:
    - Injection-mode noise generation (single key → full tensor)
    - Adaptive parameters (alpha changes per pair based on distance)
    - Per-offspring distinct randomness (via swap_mask)
    - Vectorized recombination across all pairs/offspring
    """
    num_offspring: int = struct.field(pytree_node=False, default=1)
    min_alpha: float = 0.1
    max_alpha: float = 0.8
    distance_scale: float = 1.0
    crossover_rate: float = 0.95

    @property
    def num_keys_per_atomic_operation(self) -> int:
        """Injection mode: return 0 (handled internally)"""
        return 2  # Actually needed: 1 for decision, 1 for uniform

    def _generate_noise(self, key: chex.PRNGKey, config: RealGenomeConfig) -> Tuple[chex.Array, chex.Array]:
        """
        Tier 2 - Generate full (input_length, num_offspring, ...) noise tensors.
        """
        if self.input_length <= 0 or self.num_offspring <= 0:
            raise ValueError("Set input_length and num_offspring before calling _generate_noise")

        n = int(self.input_length * self.num_offspring)
        subkeys = jax.random.split(key, n * 2).reshape((n, 2, -1))

        def per_row(k_row: chex.Array) -> Tuple[chex.Array, chex.Array]:
            k_do, k_val = k_row[0], k_row[1]
            should_cross = jax.random.bernoulli(k_do, p=self.crossover_rate)
            random_samples = jax.random.uniform(k_val, shape=config.shape, dtype=config.dtype)
            return should_cross, random_samples

        should_cross_arr, random_arr = jax.vmap(per_row)(subkeys)
        return should_cross_arr, random_arr

    def _recombine_one(
        self,
        p1: RealGenome,
        p2: RealGenome,
        noise_data: Tuple[chex.Array, chex.Array],
        config: RealGenomeConfig,
        **kwargs: Any
    ) -> RealGenome:
        """
        Tier 1 - Pure arithmetic with adaptive alpha.
        """
        should_cross, random_vals = noise_data
        dtype = config.dtype

        # Adaptive alpha based on parent distance
        parent_distance = p1.distance(p2, metric="euclidean")
        normalized_distance = jnp.tanh(parent_distance * self.distance_scale)
        adaptive_alpha = self.min_alpha + (self.max_alpha - self.min_alpha) * normalized_distance
        adaptive_alpha = jnp.array(adaptive_alpha, dtype=dtype)

        # Standard BLX-α with adaptive alpha
        diff = jnp.abs(p1.values - p2.values)
        cmin = jnp.minimum(p1.values, p2.values) - (adaptive_alpha * diff)
        cmax = jnp.maximum(p1.values, p2.values) + (adaptive_alpha * diff)

        blended_values = cmin + random_vals * (cmax - cmin)
        min_bound, max_bound = config.bounds
        blended_values = jnp.clip(blended_values, min_bound, max_bound)

        final_values = jnp.where(should_cross, blended_values, p1.values)
        return cast(RealGenome, p1.replace(values=final_values)) # cast to RealGenome but it's going to be considered no op

In [ ]:
adaptive_inj = AdaptiveAlphaCrossover_injection(
    num_offspring=2,
    min_alpha=0.1,
    max_alpha=0.8,
    distance_scale=0.5,
    crossover_rate=1.0
)

# Use existing populations
adaptive_inj = adaptive_inj.set_input_length(len(p1_pop))

print("\n" + "="*70)
print("Testing Custom AdaptiveAlphaCrossover_injection")
print("="*70)

key, subkey = jr.split(key)

# Perform injection crossover
offspring_pop = adaptive_inj(subkey, p1_pop, p2_pop, real_config)

print(f"\nParent Population 1 size: {len(p1_pop)}")
print(f"Parent Population 2 size: {len(p2_pop)}")
print(f"Offspring Population size: {len(offspring_pop)}")
print(f"Expected: {len(p1_pop) * adaptive_inj.num_offspring}")

# Verify adaptive behavior
distances = jnp.array([
    p1_pop[i].distance(p2_pop[i], metric="euclidean")
    for i in range(min(5, len(p1_pop)))
])

print(f"\nFirst 5 parent pair distances: {distances}")
print(f"\n Injection mode successfully processed {len(offspring_pop)} offspring")
print("   from pre-generated noise tensors!")


Testing Custom AdaptiveAlphaCrossover_injection

Parent Population 1 size: 500
Parent Population 2 size: 500
Offspring Population size: 1000
Expected: 1000

First 5 parent pair distances: [13.153404 15.173511 12.017388 15.016149 18.059597]

✅ Injection mode successfully processed 1000 offspring
   from pre-generated noise tensors!
